# LlamaIndex 元数据贯穿 (Metadata Propagation) 实战案例

本 Notebook 将演示如何在 LlamaIndex 中实现元数据的全链路贯穿：从源文件 -> 文档 (Documents) -> 节点 (Nodes) -> 索引 (Index) -> 查询结果 (Response)。

元数据在 RAG (Retrieval-Augmented Generation) 系统中非常重要，它可以用于：
1. **过滤 (Filtering)**：在检索时根据元数据（如年份、作者、标签）缩小搜索范围。
2. **增强上下文 (Context Enhancement)**：让 LLM 知道这段文本来自哪里（例如：“根据 2023 年的财务报告...”）。
3. **引用追踪 (Citation)**：在最终答案中给出精确的来源出处。

## 模块 1：环境安装与配置

首先安装必要的依赖包。如果你在 Colab 中运行，请执行以下代码块。

In [1]:
!pip install llama-index

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 133.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import openai

# 请在这里设置你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# 检查 API Key 是否存在
if not os.environ.get("OPENAI_API_KEY"):
    print("请设置 OPENAI_API_KEY 环境变量！")
else:
    print("OpenAI API Key 已设置。")

OpenAI API Key 已设置。


## 模块 2：生成模拟数据

为了演示元数据提取，我们需要一些具有结构化文件名或内容的文档。这里我们生成几份模拟的“公司年度财务报告”和“产品技术文档”。

我们将数据保存在 `data/simulation` 目录下。

In [3]:
import os

# 创建数据目录
data_dir = "data/simulation"
os.makedirs(data_dir, exist_ok=True)

# 模拟数据内容
files_data = {
    "finance_report_2021.txt": "2021年公司财务表现强劲，总营收增长了 15%，达到 5000 万美元。主要增长来自云服务部门。",
    "finance_report_2022.txt": "2022年市场环境充满挑战，但公司依然保持了 5% 的增长，总营收达到 5250 万美元。研发投入增加了 20%。",
    "finance_report_2023.txt": "2023年是突破性的一年，AI 产品的推出带动了 30% 的营收增长，总营收达到 6825 万美元。",
    "product_alpha_spec.txt": "Alpha 产品是一款基于 Transformer 架构的智能助手，支持多模态输入，延迟低于 100ms。发布于 2022 年。",
    "product_beta_spec.txt": "Beta 产品是专为边缘计算设计的轻量级模型，参数量仅为 1B，适合在移动设备上运行。发布于 2023 年。"
}

# 写入文件
for filename, content in files_data.items():
    with open(os.path.join(data_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"成功在 {data_dir} 下生成了 {len(files_data)} 个模拟文件。")

成功在 data/simulation 下生成了 5 个模拟文件。


In [4]:
!ls data/simulation

finance_report_2021.txt  finance_report_2023.txt  product_beta_spec.txt
finance_report_2022.txt  product_alpha_spec.txt


## 模块 3：加载数据并注入元数据

这是关键的一步。我们使用 `SimpleDirectoryReader` 加载数据，但不仅仅是加载文本，我们还要利用 `file_metadata` 参数，根据文件名自动提取元数据（例如年份、文档类型）。

In [5]:
from llama_index.core import SimpleDirectoryReader
import re

def get_meta(file_path):
    """
    根据文件路径提取元数据。
    我们假设文件名格式为：{category}_{name}_{year}.txt 或 {category}_{name}_spec.txt
    """
    filename = os.path.basename(file_path)
    meta = {}
    
    # 提取年份 (如果文件名包含 4 位数字)
    year_match = re.search(r"(\d{4})", filename)
    if year_match:
        meta["year"] = int(year_match.group(1))
    
    # 提取类别 (根据文件名关键词)
    if "finance" in filename:
        meta["category"] = "finance"
    elif "product" in filename:
        meta["category"] = "product"
    else:
        meta["category"] = "general"
        
    # 记录文件名
    meta["filename"] = filename
    return meta

# 使用 SimpleDirectoryReader 加载数据，并传入 file_metadata 函数
reader = SimpleDirectoryReader(input_dir=data_dir, file_metadata=get_meta)
documents = reader.load_data()

print(f"加载了 {len(documents)} 个文档。\n")

# 打印第一个文档查看元数据
print("=== 文档示例 (Document 0) ===")
print(f"内容: {documents[0].text}")
print(f"元数据: {documents[0].metadata}")

加载了 5 个文档。

=== 文档示例 (Document 0) ===
内容: 2021年公司财务表现强劲，总营收增长了 15%，达到 5000 万美元。主要增长来自云服务部门。
元数据: {'year': 2021, 'category': 'finance', 'filename': 'finance_report_2021.txt'}


## 模块 4：文档分块与元数据传播验证

在 LlamaIndex 中，`Document` 会被切分为更小的 `Node`。默认情况下，`Document` 的元数据会自动复制到它产生的所有 `Node` 中。我们来验证这一点。

In [6]:
from llama_index.core.node_parser import SentenceSplitter

# 创建一个解析器，这里为了演示，我们将 chunk_size 设置得很小，强制切分
parser = SentenceSplitter(chunk_size=50, chunk_overlap=10)

# 获取节点
nodes = parser.get_nodes_from_documents(documents)

print(f"从 {len(documents)} 个文档中解析出了 {len(nodes)} 个节点。\n")

# 检查前几个节点的元数据
for i, node in enumerate(nodes[:3]):
    print(f"--- 节点 {i} ---")
    print(f"内容片段: {node.get_content()[:30]}...")
    print(f"元数据: {node.metadata}")
    print(f"源文档ID: {node.ref_doc_id}")
    print("")

Metadata length (18) is close to chunk size (50). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (18) is close to chunk size (50). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (18) is close to chunk size (50). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (10) is close to chunk size (50). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (10) is close to chunk size (50). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
从 5 个文档中解析出了 11 个节点。

--- 节点 0 ---
内容片段: 2021年公司财务表现强劲，总营收增长了 15%，达到 50...
元数据: {'year': 2

可以看到，每个 `Node` 都完整继承了源 `Document` 的 `year`、`category` 和 `filename` 等元数据。这就是**元数据贯穿**的基础。

## 模块 5：构建索引、查询与结果分析

最后，我们构建向量索引，并进行查询。我们将展示检索到的结果对象中，依然保留了这些元数据，这对于后续的答案生成引用非常有用。

In [7]:
from llama_index.core import VectorStoreIndex

# 构建索引
index = VectorStoreIndex(nodes)

# 创建查询引擎
query_engine = index.as_query_engine(similarity_top_k=3)

# 执行查询
response = query_engine.query("哪一年的营收增长最快？")

print(f"回答: {response}\n")

print("=== 检索到的源节点信息 (Source Nodes) ===")
for source_node in response.source_nodes:
    print(f"相关度分数: {source_node.score:.4f}")
    print(f"文件名: {source_node.node.metadata.get('filename')}")
    print(f"年份: {source_node.node.metadata.get('year')}")
    print(f"内容: {source_node.node.get_content()[:50]}...")
    print("--------------------------------------------------")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 进阶：基于元数据的过滤查询

既然我们有了元数据，就可以在查询时进行精确过滤。例如，只查询 2023 年的文档。

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

# 设置过滤器：只查找 year == 2023 的内容
filters = MetadataFilters(
    filters=[
        MetadataFilter(key="year", value=2023)
    ]
)

filtered_query_engine = index.as_query_engine(filters=filters)

response_2023 = filtered_query_engine.query("这一年的营收情况如何？")

print(f"针对 2023 年的回答: {response_2023}\n")

print("=== 验证检索来源 ===")
for source_node in response_2023.source_nodes:
    print(f"文件名: {source_node.node.metadata.get('filename')}")
    print(f"年份: {source_node.node.metadata.get('year')}")